# PLN - Etapa Prática 1: Coleta e Pré-processamento de Dados
**Tema:** Resenhas de filmes do Letterboxd
**Equipe:** Aline Sabel e Henrique André Oneda
**Fonte de dados:** https://letterboxd.com/ (via biblioteca [`letterboxdpy`](https://github.com/nmcassa/letterboxdpy))

Este notebook implementa as dimensões avaliadas no template da disciplina:
1. **Base de dados textuais** — coleta de resenhas reais do Letterboxd, organizadas por filme.
2. **Script Python** — scraping via `letterboxdpy`, com tratamento de erros e reprodutibilidade.
3. **Limpeza e preparação dos dados** — tokenização, normalização, remoção de ruído e stopwords.
4. **Bonus round** — stemming e bases adicionais (múltiplos gêneros, para maior variedade).

> **Nota técnica sobre a biblioteca:** `Movie(slug).get_reviews()` (o método pensado para trazer
> resenhas por filme) é um stub incompleto na versão atual — sempre retorna vazio. O que **funciona de
> verdade** é `Movie(slug).popular_reviews`, preenchido automaticamente ao instanciar `Movie`: ele lê a
> seção "Popular Reviews" já presente na página do filme. Duas limitações a ter em mente:
> - **sem paginação** — só traz as resenhas exibidas naquela seção da página (não dá pra pedir mais páginas);
> - **só o primeiro parágrafo** de cada resenha é capturado pelo parser da biblioteca.
>
> Por isso, para ganhar volume e variedade (critério do template), a coleta varia os **filmes**
> (populares + vários gêneros) em vez de tentar paginar resenhas de um único filme.


## 1. Instalação de dependências

In [ ]:
# Biblioteca de scraping do Letterboxd (instalando direto do GitHub, versão mais recente)
!pip install -q git+https://github.com/nmcassa/letterboxdpy.git

# Bibliotecas de PLN e utilidades
!pip install -q nltk langdetect unidecode pandas tqdm


In [ ]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from langdetect import detect, DetectorFactory, LangDetectException

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("rslp", quiet=True)  # stemmer para português

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer, SnowballStemmer

from letterboxdpy.films import Films, get_movies_by_genre
from letterboxdpy.movie import Movie
from letterboxdpy.core.exceptions import (
    PrivateRouteError,
    ResourceNotFoundError,
    AccessDeniedError,
    InvalidResponseError,
    PageLoadError,
)

DetectorFactory.seed = 42  # torna a detecção de idioma determinística


## 2. Coleta de dados

**Justificativa da escolha da fonte:** o Letterboxd concentra um grande volume de resenhas de filmes
escritas livremente por usuários reais, com nota (0.5–5.0), texto e metadados do filme associado —
ideal para tarefas de classificação de sentimento, extração de informação e recuperação de informação
(conforme mapeado na Unidade 1 do trabalho).

**Estratégia:** montamos um conjunto diverso de filmes combinando a lista de populares com várias
listas por gênero (`get_movies_by_genre`), e para cada filme criamos um `Movie(slug)` — que já vem
com gênero, diretor, ano e nota média prontos, além das resenhas populares daquele filme
(`popular_reviews`). Isso resolve a coleta e o enriquecimento (item bônus) em uma única etapa.


In [ ]:
# --- Parâmetros de coleta (ajuste conforme o tempo disponível / limite de requisições) ---
GENEROS = [
    "action", "comedy", "drama", "horror", "romance",
    "science-fiction", "documentary", "animation",
]
MAX_POR_LISTA = 40       # filmes por lista (populares + cada gênero)
PAUSA_ENTRE_REQS = 1.0   # segundos entre requisições, para não sobrecarregar o servidor
ARQ_BRUTO = Path("resenhas_letterboxd_bruto.csv")

print("Buscando filmes populares...")
filmes = dict(Films("https://letterboxd.com/films/popular/", max=MAX_POR_LISTA).movies)

for genero in GENEROS:
    print(f"Buscando filmes do gênero '{genero}'...")
    filmes |= get_movies_by_genre(genero, max=MAX_POR_LISTA)
    time.sleep(PAUSA_ENTRE_REQS)

slugs = sorted({info["slug"] for info in filmes.values()})
print(f"\n{len(slugs)} filmes únicos reunidos (populares + {len(GENEROS)} gêneros).")


In [ ]:
def coletar_resenhas_por_filme(slugs, pausa=PAUSA_ENTRE_REQS):
    """Para cada slug de filme, instancia Movie (que já traz metadados + popular_reviews)
    e retorna uma lista de dicionários (um por resenha)."""
    registros = []
    erros = []

    for slug in tqdm(slugs, desc="Coletando por filme"):
        try:
            m = Movie(slug)
        except (PrivateRouteError, ResourceNotFoundError, AccessDeniedError,
                InvalidResponseError, PageLoadError) as e:
            erros.append({"filme_slug": slug, "erro": str(type(e).__name__)})
            continue
        except Exception as e:
            erros.append({"filme_slug": slug, "erro": f"inesperado: {e}"})
            continue

        generos = [g["name"] for g in (m.genres or [])]
        diretores = [d["name"] for d in m.crew.get("director", [])] if m.crew else []

        for rev in (m.popular_reviews or []):
            texto = rev.get("review")
            if not texto:
                continue  # sem texto (ex.: só avaliou com nota, sem escrever nada)
            registros.append({
                "filme": m.title,
                "filme_slug": slug,
                "ano_lancamento": m.year,
                "generos": generos,
                "diretores": diretores,
                "nota_media_letterboxd": m.rating,
                "usuario": (rev.get("user") or {}).get("username"),
                "nota_resenha": rev.get("rating"),
                "link": rev.get("link"),
                "resenha_original": texto,
            })

        time.sleep(pausa)

    return registros, erros

registros, erros_coleta = coletar_resenhas_por_filme(slugs)
print(f"\n{len(registros)} resenhas coletadas | {len(erros_coleta)} filmes pulados por erro.")


In [ ]:
df = pd.DataFrame(registros)
df.drop_duplicates(subset=["filme_slug", "usuario", "resenha_original"], inplace=True)
df.to_csv(ARQ_BRUTO, index=False, encoding="utf-8")
print(f"Base bruta salva em: {ARQ_BRUTO.resolve()}")
df.head()


## 3. Limpeza e pré-processamento dos dados

Etapas aplicadas ao texto de cada resenha:
1. **Remoção de ruído**: tags HTML remanescentes, URLs, quebras de linha excessivas.
2. **Detecção de idioma**: o Letterboxd é multilíngue (um dos obstáculos já identificados na Unidade 1),
   então detectamos o idioma de cada resenha para aplicar stopwords/stemming corretos.
3. **Normalização**: caixa baixa.
4. **Tokenização**: `nltk.word_tokenize`.
5. **Remoção de stopwords e pontuação** (stopwords específicas do idioma detectado).
6. **Bonus – Stemming**: `RSLPStemmer` para português e `SnowballStemmer` para inglês/outros idiomas suportados.


In [ ]:
STOPWORDS_PT = set(stopwords.words("portuguese"))
STOPWORDS_EN = set(stopwords.words("english"))

stemmer_pt = RSLPStemmer()
stemmer_en = SnowballStemmer("english")

def limpar_ruido(texto):
    """Remove HTML, URLs e espaçamento excessivo."""
    texto = re.sub(r"<[^>]+>", " ", texto)          # tags HTML
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)  # URLs
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def detectar_idioma(texto):
    try:
        return detect(texto)
    except LangDetectException:
        return "desconhecido"

def tokenizar_e_limpar(texto, idioma):
    texto = texto.lower()
    tokens = word_tokenize(texto)
    tokens = [t for t in tokens if t.isalpha()]  # remove pontuação e números

    if idioma == "pt":
        stop = STOPWORDS_PT
    elif idioma == "en":
        stop = STOPWORDS_EN
    else:
        stop = set()  # idiomas sem lista de stopwords carregada

    return [t for t in tokens if t not in stop]

def aplicar_stemming(tokens, idioma):
    if idioma == "pt":
        return [stemmer_pt.stem(t) for t in tokens]
    if idioma == "en":
        return [stemmer_en.stem(t) for t in tokens]
    return tokens  # sem stemmer disponível para o idioma


In [ ]:
tqdm.pandas()

df["resenha_limpa"] = df["resenha_original"].progress_apply(limpar_ruido)
df = df[df["resenha_limpa"].str.len() > 0].copy()  # descarta resenhas vazias após limpeza

df["idioma"] = df["resenha_limpa"].progress_apply(detectar_idioma)
df["tokens"] = df.progress_apply(lambda r: tokenizar_e_limpar(r["resenha_limpa"], r["idioma"]), axis=1)
df["tokens_stemizados"] = df.progress_apply(lambda r: aplicar_stemming(r["tokens"], r["idioma"]), axis=1)

print("Distribuição de idiomas detectados:")
print(df["idioma"].value_counts())
df[["filme", "resenha_original", "idioma", "tokens"]].head()


## 4. Dicionário de dados

| Coluna | Tipo | Descrição |
|---|---|---|
| `filme` | str | Título do filme |
| `filme_slug` | str | Slug do filme no Letterboxd |
| `ano_lancamento` | int | Ano de lançamento do filme |
| `generos` | list[str] | Gêneros do filme |
| `diretores` | list[str] | Diretor(es) do filme |
| `nota_media_letterboxd` | float | Nota média do filme na plataforma |
| `usuario` | str | Nome de usuário de quem escreveu a resenha |
| `nota_resenha` | float | Nota dada pelo usuário naquela resenha (0.5 a 5.0) |
| `link` | str | URL da resenha |
| `resenha_original` | str | Texto bruto da resenha, como coletado (primeiro parágrafo) |
| `resenha_limpa` | str | Texto após remoção de ruído (HTML, URLs, espaços) |
| `idioma` | str | Idioma detectado (`pt`, `en`, outro código ISO ou `desconhecido`) |
| `tokens` | list[str] | Tokens após normalização, tokenização e remoção de stopwords |
| `tokens_stemizados` | list[str] | Tokens após stemming (bonus) |


## 5. Exportação da base final

In [ ]:
ARQ_FINAL = Path("resenhas_letterboxd_processado.csv")

df_export = df.copy()
df_export["tokens"] = df_export["tokens"].apply(json.dumps, ensure_ascii=False)
df_export["tokens_stemizados"] = df_export["tokens_stemizados"].apply(json.dumps, ensure_ascii=False)
df_export["generos"] = df_export["generos"].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x)
df_export["diretores"] = df_export["diretores"].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x)

df_export.to_csv(ARQ_FINAL, index=False, encoding="utf-8")
print(f"Base final salva em: {ARQ_FINAL.resolve()}")
print(f"Total de resenhas no dataset final: {len(df_export)}")
print(f"Total de filmes distintos: {df_export['filme_slug'].nunique()}")


## 6. Observações e limitações

- **Volume por filme é pequeno e fixo** (a seção "Popular Reviews" da página não pagina), então o volume
  total do dataset depende de quantos filmes distintos são varridos — ajuste `MAX_POR_LISTA` e a lista de
  `GENEROS` para mais ou menos volume.
- **Só o primeiro parágrafo** de cada resenha é capturado (limitação do parser da biblioteca); resenhas
  mais longas ficam truncadas no texto original.
- A coleta depende de scraping de HTML (a lib não usa a API oficial do Letterboxd, que é fechada), então
  mudanças no site podem quebrar o parsing — se algo falhar, vale checar releases mais recentes do
  [`letterboxdpy`](https://github.com/nmcassa/letterboxdpy/releases).
- Filmes com página indisponível/erro são automaticamente pulados e listados em `erros_coleta`.
- **Alternativa com mais volume por item, porém menos variedade de filmes:** a biblioteca também tem
  `UserReviews(usuario).get_reviews()`, que retorna *todas* as resenhas escritas por um usuário (com
  paginação completa e texto integral, sem truncar em um parágrafo). Uma coleta híbrida — filmes variados
  via `Movie.popular_reviews` + aprofundamento em usuários ativos via `UserReviews` — pode ser uma boa
  extensão para o bônus "bases adicionais".
- Para a **automatização periódica** (item do bônus), este notebook pode ser agendado (ex.: cron, GitHub
  Actions) para rodar em intervalos e concatenar novos dados ao CSV final, deduplicando as linhas.
